# Projeto G1 — Desemprego no Brasil (2015–2024)

## 1. Introdução ao problema

Este projeto analisa uma base **simulada** sobre desemprego no Brasil entre 2015 e 2024. O objetivo é investigar a evolução temporal da taxa de desemprego, comparar regiões e UFs e observar possíveis relações com renda média, vagas formais e inflação.

> **Observação:** os dados são simulados e foram utilizados exclusivamente para fins acadêmicos. Portanto, os resultados não representam estatísticas oficiais.


## 2. Explicação da base de dados

A base contém registros trimestrais por unidade federativa e região. As principais variáveis são:

- `ano`, `trimestre` e `data`: dimensão temporal;
- `regiao` e `uf`: localização;
- `populacao_ativa`, `empregados` e `desempregados`: indicadores do mercado de trabalho;
- `taxa_desemprego`: taxa percentual de desemprego;
- `renda_media`: renda média;
- `setor_predominante`: setor predominante;
- `vagas_formais`: quantidade de vagas formais;
- `inflacao`: indicador de inflação;
- `nivel_risco`: classificação de risco.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
import plotly.express as px

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("../dados/simulacao_desemprego_brasil.csv")
df["data"] = pd.to_datetime(df["data"])
df.head()


## 3. Leitura dos dados

In [ ]:
print("Dimensões:", df.shape)
print("\nTipos:")
display(df.dtypes)
print("\nResumo estatístico:")
display(df.describe(include="all").T)


## 4. Limpeza e preparação

In [ ]:
# Verificação de nulos e duplicados
print("Valores nulos por coluna:")
display(df.isna().sum())
print("Duplicados:", df.duplicated().sum())

# Padronização e validações
df["regiao"] = df["regiao"].str.strip()
df["uf"] = df["uf"].str.strip().str.upper()
df["setor_predominante"] = df["setor_predominante"].str.strip()
df["nivel_risco"] = df["nivel_risco"].str.strip()

# Validação lógica: população ativa = empregados + desempregados
df["saldo_populacao_ativa"] = df["populacao_ativa"] - df["empregados"] - df["desempregados"]

print("Maior saldo de validação:", df["saldo_populacao_ativa"].abs().max())
print("Período:", df["data"].min().date(), "até", df["data"].max().date())
print("UFs:", df["uf"].nunique(), "| Regiões:", df["regiao"].nunique())

# Tratamento da coluna auxiliar após a validação
df = df.drop(columns="saldo_populacao_ativa")


## 5. Engenharia de atributos

In [ ]:
df["periodo"] = df["data"].dt.to_period("Q").astype(str)
df["taxa_emprego"] = (df["empregados"] / df["populacao_ativa"]) * 100
df["vagas_por_mil_ativos"] = (df["vagas_formais"] / df["populacao_ativa"]) * 1000
df["desempregados_milhoes"] = df["desempregados"] / 1_000_000

df[["periodo","taxa_emprego","vagas_por_mil_ativos","desempregados_milhoes"]].head()


## 6. Análise exploratória

In [ ]:
print("Anos:", sorted(df["ano"].unique()))
print("Trimestres:", sorted(df["trimestre"].unique()))
print("Distribuição por região:")
display(df["regiao"].value_counts())
print("Distribuição por nível de risco:")
display(df["nivel_risco"].value_counts())


### Evolução temporal

In [ ]:
temporal = df.groupby("data", as_index=False)["taxa_desemprego"].mean()

plt.figure(figsize=(12,5))
sns.lineplot(data=temporal, x="data", y="taxa_desemprego", marker="o")
plt.title("Evolução da taxa média de desemprego")
plt.xlabel("Período")
plt.ylabel("Taxa de desemprego (%)")
plt.tight_layout()
plt.show()


### Comparação entre regiões

In [ ]:
regional = (df.groupby("regiao", as_index=False)["taxa_desemprego"]
             .mean()
             .sort_values("taxa_desemprego", ascending=False))

plt.figure(figsize=(10,5))
sns.barplot(data=regional, x="taxa_desemprego", y="regiao")
plt.title("Taxa média de desemprego por região")
plt.xlabel("Taxa média (%)")
plt.ylabel("Região")
plt.tight_layout()
plt.show()

display(regional.round(2))


### Comparação por UF

In [ ]:
uf_ranking = (df.groupby("uf", as_index=False)["taxa_desemprego"]
               .mean()
               .sort_values("taxa_desemprego", ascending=False))

display(uf_ranking.round(2))

plt.figure(figsize=(10,7))
sns.barplot(data=uf_ranking, x="taxa_desemprego", y="uf")
plt.title("Ranking das UFs pela taxa média de desemprego")
plt.xlabel("Taxa média (%)")
plt.ylabel("UF")
plt.tight_layout()
plt.show()


### Indicadores por setor

In [ ]:
setor = (df.groupby("setor_predominante", as_index=False)
          .agg(taxa_media=("taxa_desemprego","mean"),
               renda_media=("renda_media","mean"),
               vagas=("vagas_formais","sum"))
          .sort_values("taxa_media", ascending=False))
display(setor.round(2))


## 7. KPIs

In [ ]:
kpis = {
    "Taxa média de desemprego (%)": df["taxa_desemprego"].mean(),
    "Desempregados acumulados": df["desempregados"].sum(),
    "Renda média (R$)": df["renda_media"].mean(),
    "Vagas formais acumuladas": df["vagas_formais"].sum(),
}
for nome, valor in kpis.items():
    if "Taxa" in nome:
        print(f"{nome}: {valor:.2f}")
    elif "Renda" in nome:
        print(f"{nome}: R$ {valor:,.2f}")
    else:
        print(f"{nome}: {valor:,.0f}")


## 8. Gráficos e relações estatísticas

In [ ]:
corr = df[["taxa_desemprego","renda_media","vagas_formais","inflacao"]].corr()
plt.figure(figsize=(8,5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlação")
plt.tight_layout()
plt.show()


In [ ]:
fig = px.scatter(
    df, x="renda_media", y="taxa_desemprego", color="regiao",
    hover_data=["uf","ano","trimestre"],
    title="Renda média × taxa de desemprego",
    labels={"renda_media":"Renda média (R$)", "taxa_desemprego":"Taxa de desemprego (%)"}
)
fig.show()


## 9. Interpretação dos resultados

Na base simulada, a taxa média varia entre as regiões. O ranking regional permite identificar quais regiões apresentam maior e menor desemprego médio no período analisado. A série temporal permite observar oscilações entre os trimestres e anos.

A análise por UF detalha as diferenças dentro das regiões. Os indicadores de renda, vagas formais e inflação foram comparados por meio de correlação para verificar associações lineares. Essas correlações não devem ser interpretadas como relações de causa e efeito.

A classificação de risco complementa a análise ao separar as observações em níveis Baixo, Médio, Alto e Crítico.


## 10. Conclusão

O projeto demonstra um fluxo completo de análise de dados: leitura, validação, limpeza, preparação, engenharia de atributos, análise exploratória, criação de KPIs e visualização.

O dashboard Streamlit amplia a análise com filtros múltiplos, indicadores dinâmicos, tabelas, gráficos interativos, análise temporal e mapa por UF. O projeto também utiliza SQLite com SQLAlchemy para persistência dos dados.

Como a base é simulada, a principal conclusão acadêmica é sobre o uso das técnicas de análise e visualização, e não sobre o comportamento real do mercado de trabalho brasileiro.
